# 60. Tracking the rocket with two small infrasound arrays (feasibility)

Two sites have several Gems close together and form small arrays:

* **B20** (7.6 km SE of the pad): B20 `HDF`, B20 `10.HDI` (~1 m away), B20W (~27 m W), B20S (~35 m S);
  aperture ~35 m, suited to ~2–10 Hz at 100 sps. Gem 363 (`1R.UNK..HD1`) is included at the position
  from its own GPS log (~8 m from B20), whatever its final station code (`INCLUDE_UNK`).
* **B24** (5.5 km SSW): B24 `HDF` and `HDI` (~6 m apart), B24LT (~205 m W), B24RE (~250 m WNW);
  aperture ~250 m, suited to lower frequencies.

In sliding windows, pairwise cross-correlation lags are fitted with a plane wave to give the
**back-azimuth** (direction toward the source) and **trace velocity**; trace velocity gives the
ray's **elevation angle** (cos e = c / v_trace). These are compared with the forward model
(notebook 40 geometry), and the two arrays' back-azimuths are intersected to locate the source in map
view.

Simplifications (feasibility only): plane waves, straight rays, no wind; the intersection pairs windows
with the same arrival time at both arrays (the two arrays are at similar distances once the rocket is
high), and elevation from a 35 m array is coarse.

Outputs (`cfg.OUT_DIR`): `array_estimates.csv`, `array_triangulation.csv`, `array_backazimuth.png`,
`array_map.png`.

In [ ]:
import sys
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy.geodetics import gps2dist_azimuth

import artemis2_config as cfg
if str(cfg.ROCKET_MODEL_DIR) not in sys.path:
    sys.path.append(str(cfg.ROCKET_MODEL_DIR))
from rocket_infrasound_model.trajectory import GravityTurnParameters, gravity_turn_trajectory

L = cfg.LAUNCHTIME
OUT = cfg.OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'axes.grid': True, 'grid.color': '#dddddd', 'grid.linewidth': 0.6,
                     'axes.spines.top': False, 'axes.spines.right': False})
C_SOUND = 343.0
AZ = cfg.LAUNCH_AZIMUTH_DEG
INCLUDE_UNK = True     # Gem 363: GPS log puts it ~8 m from B20 (notebook 10)
T_RANGE = (0, 180)               # s after launch (arrival time)
ARRAYS = {
    'B20': dict(ids=['1R.B20..HDF', '1R.B20.10.HDI', '1R.B20W..HDF', '1R.B20S..HDF'] + (['1R.UNK..HD1'] if INCLUDE_UNK else []),
                band=(2.0, 10.0), win=2.0, step=0.5),
    'B24': dict(ids=['1R.B24..HDF', '1R.B24..HDI', '1R.B24LT..HDF', '1R.B24RE..HDF'],
                band=(0.5, 3.0), win=4.0, step=1.0),
}
CC_MIN = 0.7                     # minimum mean pairwise correlation to accept a window

## 1. Element positions from the Gem GPS logs (notebook 10)

In [ ]:
gps = pd.read_csv(cfg.QC_DIR / 'gem_gps_coordinates.csv')
mapping = pd.read_csv(cfg.GEM_MAPPING_CSV)
gps = gps.merge(mapping, left_on='SN', right_on='serial', how='left')
pads = pd.read_excel(cfg.STATION_XLS, sheet_name='launchpads_2026').set_index('pad')
plat, plon = pads.loc[cfg.PAD, ['lat', 'lon']]

def enu(lat, lon):
    d, az, _ = gps2dist_azimuth(plat, plon, lat, lon)
    return d * np.sin(np.radians(az)), d * np.cos(np.radians(az))

pos = {}
for _, r in gps.dropna(subset=['id']).iterrows():
    pos[r.id] = np.array(enu(r.lat, r.lon))
for name, a in ARRAYS.items():
    xy = np.array([pos[i] for i in a['ids']])
    a['xy'] = xy
    a['centre'] = xy.mean(axis=0)
    print(name, 'centre (E, N) from pad:', a['centre'].round(0), 'm')
    for i, p in zip(a['ids'], xy - a['centre']):
        print(f'   {i:16s} {p[0]:8.1f} {p[1]:8.1f} m')

## 2. Data (Gem correction; no element of either array overflowed, notebook 20)

In [ ]:
raw = cfg.get_group_stream('BSU').merge()
st = cfg.instrument_correct(raw.select(channel='HD?'))

## 3. Plane-wave fits in sliding windows

In [ ]:
def xcorr_lag(a, b, maxlag, up=10):
    """Lag (samples, fractional) of b relative to a, and normalised correlation, by parabolic interpolation."""
    n = len(a)
    c = np.correlate(b - b.mean(), a - a.mean(), mode='full') / (n * a.std() * b.std())
    mid = n - 1
    lo, hi = mid - maxlag, mid + maxlag + 1
    seg = c[lo:hi]
    k = int(np.argmax(seg))
    if 0 < k < len(seg) - 1:
        y0, y1, y2 = seg[k - 1], seg[k], seg[k + 1]
        d = 0.5 * (y0 - y2) / (y0 - 2 * y1 + y2)
    else:
        d = 0.0
    return (k - maxlag) + d, seg[k]

def plane_wave(name, a):
    traces = []
    for sid in a['ids']:
        tr = st.select(id=sid)[0].copy()
        tr.filter('bandpass', freqmin=a['band'][0], freqmax=a['band'][1], corners=4, zerophase=True)
        traces.append(tr.slice(L + T_RANGE[0] - 5, L + T_RANGE[1] + 5))
    fs = traces[0].stats.sampling_rate
    n = min(t.stats.npts for t in traces)
    data = np.array([t.data[:n] for t in traces])
    tt = traces[0].times() + (traces[0].stats.starttime - L)
    rel = a['xy'] - a['centre']
    maxbase = max(np.linalg.norm(rel[i] - rel[j]) for i, j in itertools.combinations(range(len(rel)), 2))
    maxlag = int(np.ceil(1.2 * maxbase / 300.0 * fs)) + 1
    nw, ns = int(a['win'] * fs), int(a['step'] * fs)
    rows = []
    for i0 in range(0, n - nw, ns):
        G, d, w, ccs = [], [], [], []
        for i, j in itertools.combinations(range(len(rel)), 2):
            lag, cc = xcorr_lag(data[i, i0:i0 + nw], data[j, i0:i0 + nw], maxlag)
            G.append(rel[j] - rel[i]); d.append(lag / fs); w.append(max(cc, 0)); ccs.append(cc)
        G, d, w = np.array(G), np.array(d), np.array(w)
        if np.mean(ccs) < CC_MIN:
            continue
        sl, *_ = np.linalg.lstsq(G * w[:, None], d * w, rcond=None)
        res = d - G @ sl
        vtr = 1 / np.linalg.norm(sl)
        rows.append(dict(array=name, t=tt[i0 + nw // 2], baz_deg=np.degrees(np.arctan2(-sl[0], -sl[1])) % 360,
                         v_trace=vtr, elev_deg=np.degrees(np.arccos(min(1.0, C_SOUND / vtr))),
                         mean_cc=np.mean(ccs), rms_resid_ms=1e3 * np.sqrt(np.mean(res ** 2)),
                         rms_power=data[:, i0:i0 + nw].std()))
    return pd.DataFrame(rows)

est = pd.concat([plane_wave(n, a) for n, a in ARRAYS.items()], ignore_index=True)
est.to_csv(OUT / 'array_estimates.csv', index=False)
print(est.groupby('array').agg(n=('t', 'size'), cc=('mean_cc', 'median'), resid_ms=('rms_resid_ms', 'median'),
                               v_trace=('v_trace', 'median')).round(2))

## 4. Model predictions at each array

In [ ]:
TSM = np.arange(0, 200.01, 0.1)
s = gravity_turn_trajectory(TSM, GravityTurnParameters())
src_e, src_n, src_z = s['x_m'] * np.sin(np.radians(AZ)), s['x_m'] * np.cos(np.radians(AZ)), s['z_m']

def predicted(centre):
    dx, dy = src_e - centre[0], src_n - centre[1]
    R = np.sqrt(dx ** 2 + dy ** 2 + src_z ** 2)
    tobs = TSM + R / C_SOUND
    return pd.DataFrame(dict(t=tobs, t_s=TSM, baz_deg=np.degrees(np.arctan2(dx, dy)) % 360,
                             elev_deg=np.degrees(np.arcsin(src_z / R)), alt_km=src_z / 1000))

pred = {n: predicted(a['centre']) for n, a in ARRAYS.items()}

fig, ax = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for col, (name, a) in enumerate(ARRAYS.items()):
    e, p = est[est.array == name], pred[name]
    p = p[(p.t >= T_RANGE[0]) & (p.t <= T_RANGE[1])]
    ax[0, col].scatter(e.t, e.baz_deg, s=8, c=e.mean_cc, cmap='viridis', vmin=CC_MIN, vmax=1)
    ax[0, col].plot(p.t, p.baz_deg, color='#eb6834', lw=1.2, label=f'model (azimuth {AZ:g}°)')
    ax[0, col].set(ylabel='Back-azimuth (deg)', title=f"{name}: {a['band'][0]:g}-{a['band'][1]:g} Hz, "
                                                       f"{a['win']:g}-s windows")
    ax[0, col].legend(frameon=False, fontsize=8)
    ax[1, col].scatter(e.t, e.elev_deg, s=8, c=e.mean_cc, cmap='viridis', vmin=CC_MIN, vmax=1)
    ax[1, col].plot(p.t, p.elev_deg, color='#eb6834', lw=1.2)
    ax[1, col].set(ylabel='Elevation angle (deg)', xlabel='Arrival time after launch (s)', ylim=(-2, 90))
fig.suptitle('Array estimates (dots, coloured by mean correlation) vs straight-ray model', x=0.01, ha='left')
fig.tight_layout()
fig.savefig(OUT / 'array_backazimuth.png', dpi=200)

## 5. Map-view triangulation

In [ ]:
def ray_intersection(p1, b1, p2, b2):
    d1 = np.array([np.sin(np.radians(b1)), np.cos(np.radians(b1))])
    d2 = np.array([np.sin(np.radians(b2)), np.cos(np.radians(b2))])
    A = np.column_stack([d1, -d2])
    if abs(np.linalg.det(A)) < 0.1:          # nearly parallel rays
        return None
    h1, h2 = np.linalg.solve(A, p2 - p1)
    if h1 <= 0 or h2 <= 0:
        return None
    return p1 + h1 * d1, h1, h2

e20 = est[est.array == 'B20'].sort_values('t')
e24 = est[est.array == 'B24'].sort_values('t')
rows = []
for _, r in e20.iterrows():
    near = e24[np.abs(e24.t - r.t) <= ARRAYS['B24']['step']]
    if near.empty:
        continue
    b24 = np.degrees(np.angle(np.mean(np.exp(1j * np.radians(near.baz_deg)))))
    out = ray_intersection(ARRAYS['B20']['centre'], r.baz_deg, ARRAYS['B24']['centre'], b24 % 360)
    if out is None:
        continue
    p, h1, h2 = out
    el = np.nanmedian([r.elev_deg, near.elev_deg.median()])
    z = np.mean([h1, h2]) * np.tan(np.radians(el))
    R = np.sqrt(np.mean([h1, h2]) ** 2 + z ** 2)
    rows.append(dict(t_arrival=r.t, east_m=p[0], north_m=p[1], range_B20_m=h1, range_B24_m=h2,
                     alt_m_from_elev=z, t_source_est=r.t - R / C_SOUND))
tri = pd.DataFrame(rows)
tri.to_csv(OUT / 'array_triangulation.csv', index=False)
print(len(tri), 'intersections')
display(tri.round(0).head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
stations = cfg.station_table()
ax.scatter(stations.east_m / 1000, stations.north_m / 1000, marker='^', color='#888888', s=25)
for sname, r in stations.iterrows():
    ax.annotate(sname, (r.east_m / 1000, r.north_m / 1000), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.plot(0, 0, marker='*', color='#333333', ms=12)
ax.plot(src_e[TSM <= 120] / 1000, src_n[TSM <= 120] / 1000, color='#eb6834', lw=1.5, label=f'model ground track (az {AZ:g}°)')
if len(tri):
    sc = ax.scatter(tri.east_m / 1000, tri.north_m / 1000, c=tri.t_arrival, s=12, cmap='viridis', zorder=3)
    fig.colorbar(sc, ax=ax, label='Arrival time (s)', shrink=0.7)
for name, a in ARRAYS.items():
    ax.plot(*(a['centre'] / 1000), 'o', color='#2a78d6', ms=7)
ax.set(xlabel='East of LC39B (km)', ylabel='North of LC39B (km)', aspect='equal',
       title='Back-azimuth intersections, B20 and B24 arrays', xlim=(-8, 20), ylim=(-10, 10))
ax.legend(frameon=False, fontsize=8, loc='upper left')
fig.tight_layout()
fig.savefig(OUT / 'array_map.png', dpi=200)